# TextSplitter的源码剖析

主要讲一下内部的三个方法：

方法1：
split_text(self, text: str) -> list[str]:
> 传入的参数类型：文本内容（或字符串），返回值类型：字符串列表
>
> 此方法是抽象方法，具体的实现细节由子类来决定

方法2：
create_documents(self, texts: list[str],...) -> list[Document]:
> 传入的参数类型：字符串列表，返回值类型：Document对象列表
>
> 此方法的底层调用了split_text()，即将参数中的每一个字符串都传入split_text()中执行，得到的字符串列表中，将字符串封装为Document对象，就构成了list[Document]。


方法3：
split_documents(self, documents: Iterable[Document]) -> list[Document]:
> 传入的参数类型：Document对象列表，返回值类型：Document对象列表
>
> 此方法的底层调用了create_documents(),将参数中的每一个Document对象，提取其page_content字段，则构成了字符串列表，然后调用方法2即可。

# 具体文档切分器的使用


## 1、CharacterTextSplitter：Split by character

举例1：字符串文本的分割

In [1]:
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter

# 2.示例文本
text = """
LangChain 是一个用于开发由语言模型驱动的应用程序的框架的。它提供了一套工具和抽象，使开发者能够更容易地构建复杂的应用程序。
"""

splitter = CharacterTextSplitter(
    chunk_size=50, # 每块大小
     chunk_overlap=5,# 块与块之间的重复字符数
     separator=""   # 设置为空字符串时，表示禁用分隔符优先
)

# 4.分割文本
texts = splitter.split_text(text)

# 5.打印结果
for i, chunk in enumerate(texts):
    print(f"块 {i+1}:长度：{len(chunk)}")
    print(chunk)
    print("-" * 50)

块 1:长度：49
LangChain 是一个用于开发由语言模型驱动的应用程序的框架的。它提供了一套工具和抽象，使开发
--------------------------------------------------
块 2:长度：22
象，使开发者能够更容易地构建复杂的应用程序。
--------------------------------------------------


In [3]:
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter

# 2.定义要分割的文本
text = "这是一个示例文本啊。我们将使用CharacterTextSplitter将其分割成小块。分割基于字符数。"

# text = """
# LangChain 是一个用于开发由语言模型。驱动的应用程序的框架的。它提供了一套工具和抽象。使开发者能够更容易地构建复杂的应用程序。
# """

# 3.定义分割器实例
text_splitter = CharacterTextSplitter(
    chunk_size=30,   # 每个块的最大字符数
    chunk_overlap=5, # 块之间的重叠字符数
    separator="。",  # 按句号分割优先
)

# 4.开始分割
chunks = text_splitter.split_text(text)

# 5.打印效果
for  i,chunk in enumerate(chunks):
    print(f"块 {i + 1}:长度：{len(chunk)}")
    print(chunk)
    print("-"*50)


Created a chunk of size 33, which is longer than the specified 30


块 1:长度：9
这是一个示例文本啊
--------------------------------------------------
块 2:长度：33
我们将使用CharacterTextSplitter将其分割成小块
--------------------------------------------------
块 3:长度：7
分割基于字符数
--------------------------------------------------


In [4]:
# 1.导入相关依赖
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2.定义RecursiveCharacterTextSplitter分割器对象
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=10,
    chunk_overlap=0,
    add_start_index=True,
)

# 3.定义拆分的内容
text="LangChain框架特性\n\n多模型集成(GPT/Claude)\n记忆管理功能\n链式调用设计。文档分析场景示例：需要处理PDF/Word等格式。"

# 4.拆分器分割
paragraphs = text_splitter.split_text(text)

for i,chunk in enumerate(paragraphs):
    print(f"块{i + 1},长度：{len(chunk)}")
    print(chunk)
    print('-' * 50)

块1,长度：10
LangChain框
--------------------------------------------------
块2,长度：3
架特性
--------------------------------------------------
块3,长度：9
多模型集成(GPT
--------------------------------------------------
块4,长度：8
/Claude)
--------------------------------------------------
块5,长度：6
记忆管理功能
--------------------------------------------------
块6,长度：9
链式调用设计。文档
--------------------------------------------------
块7,长度：10
分析场景示例：需要处
--------------------------------------------------
块8,长度：10
理PDF/Word等
--------------------------------------------------
块9,长度：3
格式。
--------------------------------------------------


In [5]:
# 1.导入相关依赖
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2.定义RecursiveCharacterTextSplitter分割器对象
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=10,
    chunk_overlap=0,
    add_start_index=True,
)

# 3.定义分割的内容
# text="LangChain框架特性\n\n多模型集成(GPT/Claude)\n记忆管理功能\n链式调用设计。文档分析场景示例：需要处理PDF/Word等格式。"

list=["LangChain框架特性\n\n多模型集成(GPT/Claude)\n记忆管理功能\n链式调用设计。文档分析场景示例：需要处理PDF/Word等格式。"]

# 4.分割器分割
# create_documents()：形参是字符串列表，返回值是Document的列表
paragraphs = text_splitter.create_documents(list)


for para in paragraphs:
    print(para)
    print('-------')

page_content='LangChain框' metadata={'start_index': 0}
-------
page_content='架特性' metadata={'start_index': 10}
-------
page_content='多模型集成(GPT' metadata={'start_index': 15}
-------
page_content='/Claude)' metadata={'start_index': 24}
-------
page_content='记忆管理功能' metadata={'start_index': 33}
-------
page_content='链式调用设计。文档' metadata={'start_index': 40}
-------
page_content='分析场景示例：需要处' metadata={'start_index': 49}
-------
page_content='理PDF/Word等' metadata={'start_index': 59}
-------
page_content='格式。' metadata={'start_index': 69}
-------
